# Phase 2: Temporal Feature Expansion

## Goal

Improve 60-minute-ahead household power forecasting by adding features that describe
recent changes, trends, volatility, and local extremes in power consumption.

The first modeling phase showed that:

- Histogram Gradient Boosting was the strongest validation model.
- The baseline PyTorch MLP performed close to HGB but did not outperform it.
- The neural network tended to underpredict large power-consumption events.
- Changing the neural-network loss function and dropout did not improve performance.

Instead of continuing to tune the model architecture, Phase 2 will test whether
better temporal information improves forecasting performance.

## Hypothesis

The existing lag and rolling-average features describe past power levels, but they
do not explicitly describe whether consumption is currently rising, falling, or
becoming unusually volatile.

Features representing recent changes, ranges, maxima, and variability may help the
models identify developing high-consumption periods.

## Experimental Plan

1. Load the validated minute-level dataset.
2. Recreate the existing lag, rolling, calendar, and target features.
3. Add new temporal features describing:
   - recent power changes
   - recent volatility
   - recent maximum power
   - recent minimum power
   - recent power range
4. Rebuild the modeling dataset without temporal leakage.
5. Recreate the same chronological train/validation/test split.
6. Retrain Histogram Gradient Boosting using the expanded feature set.
7. Retrain the baseline MLP architecture using the expanded feature set.
8. Compare validation performance against the original Phase 1 results.
9. Keep the test set untouched until Phase 2 model selection is complete.

## Phase 1 Benchmarks

- Histogram Gradient Boosting RMSE: 0.7885
- Baseline Neural Network RMSE: approximately 0.7983

The primary question is whether the expanded temporal features improve validation
RMSE beyond these benchmarks.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

data_path = Path(
    "../data/processed/household_power_validated.parquet"
)

df = pd.read_parquet(data_path)

df = (
    df.sort_values("datetime")
      .reset_index(drop=True)
)

print(df.shape)
print(df["datetime"].min())
print(df["datetime"].max())

(2075259, 10)
2006-12-16 17:24:00
2010-11-26 21:02:00


In [3]:
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [4]:
lag_minutes = [
    1,
    5,
    15,
    30,
    60,
    120,
    1380
]

for lag in lag_minutes:
    df[f"power_lag_{lag}"] = (
        df["Global_active_power"].shift(lag)
    )

In [5]:
df["rolling_mean_15"] = (
    df["Global_active_power"]
    .rolling(15)
    .mean()
)

df["rolling_mean_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .mean()
)

df["rolling_std_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .std()
)

df["rolling_mean_1440"] = (
    df["Global_active_power"]
    .rolling(1440)
    .mean()
)

In [6]:
df["power_target_60"] = (
    df["Global_active_power"].shift(-60)
)

In [7]:
# change features

# power_change_1  > 0 → power increased during the last minute
# power_change_15 > 0 → power increased over the last 15 minutes
# power_change_60 < 0 → power decreased over the last hour

df["power_change_1"] = (
    df["Global_active_power"]
    - df["power_lag_1"]
)

df["power_change_5"] = (
    df["Global_active_power"]
    - df["power_lag_5"]
)

df["power_change_15"] = (
    df["Global_active_power"]
    - df["power_lag_15"]
)

df["power_change_60"] = (
    df["Global_active_power"]
    - df["power_lag_60"]
)

In [8]:
# volatility and extreme-value features

df["rolling_std_15"] = (
    df["Global_active_power"]
    .rolling(15)
    .std()
)

df["rolling_max_15"] = (
    df["Global_active_power"]
    .rolling(15)
    .max()
)

df["rolling_max_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .max()
)

df["rolling_min_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .min()
)

In [9]:
# range feature

df["rolling_range_60"] = (
    df["rolling_max_60"]
    - df["rolling_min_60"]
)

In [10]:
# rolling_std_15
#     → how unstable power has been recently

# rolling_max_15
#     → recent short-term peak

# rolling_max_60
#     → highest power observed during the past hour

# rolling_min_60
#     → lowest power observed during the past hour

# rolling_range_60
#     → how widely consumption has moved during the past hour

In [11]:
# check new features

new_features = [
    "power_change_1",
    "power_change_5",
    "power_change_15",
    "power_change_60",
    "rolling_std_15",
    "rolling_max_15",
    "rolling_max_60",
    "rolling_min_60",
    "rolling_range_60"
]

df[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
power_change_1,2049208.0,-0.000011,0.267337,-4.338,-0.0080,0.000000,0.00600,5.058000
power_change_5,2049122.0,-0.000046,0.539351,-7.090,-0.0700,-0.002000,0.06800,7.610000
power_change_15,2048967.0,-0.000119,0.781559,-7.060,-0.1300,-0.004000,0.11000,7.942000
power_change_60,2048439.0,-0.000333,1.064140,-9.206,-0.2500,-0.004000,0.19000,9.026000
rolling_std_15,2048290.0,0.218959,0.311600,0.000,0.0383,0.064093,0.29428,3.162321
rolling_max_15,2048290.0,1.449291,1.342729,0.078,0.3940,1.062000,1.98200,11.122000
rolling_max_60,2045179.0,2.076395,1.642923,0.188,0.8500,1.540000,3.04400,11.122000
rolling_min_60,2045179.0,0.624049,0.605295,0.076,0.2160,0.294000,1.18800,5.950000
rolling_range_60,2045179.0,1.452346,1.354854,0.000,0.2780,0.968000,2.23400,9.304000


In [12]:
# check relationship with the target

correlations = (
    df[
        new_features
        + ["power_target_60"]
    ]
    .corr()["power_target_60"]
    .sort_values(ascending=False)
)

correlations

power_target_60     1.000000
rolling_max_15      0.493902
rolling_max_60      0.472838
rolling_min_60      0.434143
rolling_range_60    0.379408
rolling_std_15      0.300563
power_change_60     0.163727
power_change_15     0.034619
power_change_5      0.028329
power_change_1      0.011349
Name: power_target_60, dtype: float64

In [13]:
## Build the Phase 2 Modeling Dataset

In [14]:
phase2_feature_cols = [
    # Current electrical measurements
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    # Original lag features
    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    # Calendar features
    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    # Original rolling features
    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440",

    # Phase 2 change features
    "power_change_1",
    "power_change_5",
    "power_change_15",
    "power_change_60",

    # Phase 2 volatility / extreme features
    "rolling_std_15",
    "rolling_max_15",
    "rolling_max_60",
    "rolling_min_60",
    "rolling_range_60"
]

target_col = "power_target_60"

print("Number of Phase 2 features:", len(phase2_feature_cols))

Number of Phase 2 features: 31


In [15]:
phase2_df = (
    df[
        ["datetime"]
        + phase2_feature_cols
        + [target_col]
    ]
    .dropna()
    .reset_index(drop=True)
)

In [16]:
print(f"Original timeline rows: {len(df):,}")
print(f"Phase 2 usable rows:    {len(phase2_df):,}")
print(
    f"Rows removed:           "
    f"{len(df) - len(phase2_df):,}"
)

print("\nFinal shape:", phase2_df.shape)

Original timeline rows: 2,075,259
Phase 2 usable rows:    1,950,244
Rows removed:           125,015

Final shape: (1950244, 33)


In [17]:
## Validate the Expanded Feature Dataset

In [18]:
missing = phase2_df.isna().sum()

print(
    "Total missing values:",
    missing.sum()
)

print(
    missing[missing > 0]
)

Total missing values: 0
Series([], dtype: int64)


In [19]:
print(
    "Chronological:",
    phase2_df["datetime"].is_monotonic_increasing
)

Chronological: True


In [20]:
print(
    "Duplicate timestamps:",
    phase2_df["datetime"].duplicated().sum()
)

Duplicate timestamps: 0


In [21]:
print(
    "Feature count:",
    len(phase2_feature_cols)
)

print(
    "Expected dataframe columns:",
    1 + len(phase2_feature_cols) + 1
)

print(
    "Actual dataframe columns:",
    phase2_df.shape[1]
)

Feature count: 31
Expected dataframe columns: 33
Actual dataframe columns: 33


In [22]:
# inspect

phase2_df[
    [
        "datetime",
        "Global_active_power",
        "power_change_15",
        "rolling_std_15",
        "rolling_max_60",
        "rolling_range_60",
        "power_target_60"
    ]
].head()

,datetime,Global_active_power,power_change_15,rolling_std_15,rolling_max_60,rolling_range_60,power_target_60
0,2006-12-17 17:23:00,3.238,-0.422,0.199423,4.912,2.19,3.612
1,2006-12-17 17:24:00,3.390,-0.196,0.206197,4.912,2.19,3.612
2,2006-12-17 17:25:00,3.360,-0.196,0.213867,4.912,2.19,3.576
3,2006-12-17 17:26:00,3.400,-0.204,0.217765,4.912,2.19,3.552
4,2006-12-17 17:27:00,3.396,-0.332,0.215561,4.912,2.19,3.550


In [23]:
print(
    "Negative 60-minute ranges:",
    (phase2_df["rolling_range_60"] < 0).sum()
)

Negative 60-minute ranges: 0


In [24]:
## Save the Phase 2 Feature Dataset

In [25]:
output_path = Path(
    "../data/processed/"
    "household_power_features_phase2.parquet"
)

phase2_df.to_parquet(
    output_path,
    index=False
)

print("Saved to:", output_path)

Saved to: ../data/processed/household_power_features_phase2.parquet


In [26]:
# verify

phase2_check = pd.read_parquet(output_path)

print("Reloaded shape:", phase2_check.shape)
print(
    "Reload matches original:",
    phase2_check.shape == phase2_df.shape
)

Reloaded shape: (1950244, 33)
Reload matches original: True
